In [3]:
import logging
import pandas as pd
import matplotlib.pyplot as plt

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(message)s"
)

file_clean = "cleaned_generation_2022.xlsx"
sheet_name = "Generation"

df = pd.read_excel(file_clean, sheet_name=sheet_name)

# --- Tech → friendly labels ---
tech_label_map = {
    "Hidraulica_Pasada":       "Hydro-ROR",
    "Hidraulica_Embalse":      "Hydro-Res",
    "Termica_Turbovapor":      "Therm-Steam",
    "Termica_Turbogas":        "Therm-GT",
    "Termica_MCI":             "Therm-ICE",
    "Biomasa_Turbovapor":      "Biomass-Steam",
    "Eolica_Eolica":           "Wind",
    "Biogas_MCI":              "Biogas-ICE",
    "Solar_Solar":             "Solar-PV",
}

group = (
    df.groupby("Technology")["Potencia_Efectiva_MW"]
      .sum()
      .rename("Effective_Power_MW")
      .reset_index()
)

group["Label"] = group["Technology"].map(tech_label_map).fillna(group["Technology"])

# --- Colors per tech family ---
color_map = {
    # Hydro – blue
    "Hidraulica_Pasada":   "#1f77b4",  # Hydro-ROR
    "Hidraulica_Embalse":  "#4fa3ff",  # Hydro-Res
    # Thermal – red/orange
    "Termica_Turbovapor":  "#d62728",
    "Termica_Turbogas":    "#ff7f0e",
    "Termica_MCI":         "#b22222",
    # Biomass/Biogas – green
    "Biomasa_Turbovapor":  "#2ca02c",
    "Biogas_MCI":          "#228b22",
    # Wind – purple
    "Eolica_Eolica":       "#9467bd",
    # Solar – yellow
    "Solar_Solar":         "#ffd700",
}

colors = [color_map.get(t, "#7f7f7f") for t in group["Technology"]]

# --- Compute percentages for legend text ---
total_power = group["Effective_Power_MW"].sum()
group["Percent"] = group["Effective_Power_MW"] / total_power * 100

legend_labels = [
    f"{row.Label} ({row.Effective_Power_MW:.1f} MW, {row.Percent:.1f}%)"
    for _, row in group.iterrows()
]

# --- Plot: pie with no on-wedge labels, legend on the side ---
plt.style.use("default")

fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

wedges, _ = ax.pie(
    group["Effective_Power_MW"],
    labels=None,             # <- no labels on wedges to avoid clutter
    colors=colors,
    startangle=90,
    counterclock=False,
    wedgeprops=dict(edgecolor="white", linewidth=0.8)
)

ax.set_title(
    "Installed Effective Power by Technology",
    fontsize=11,
    pad=12
)

ax.axis("equal")  # keep pie circular

# Legend on the right
ax.legend(
    wedges,
    legend_labels,
    title="Technologies",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    fontsize=8,
    title_fontsize=9,
    borderaxespad=0.0,
)

# Note at the bottom
plt.figtext(
    0.5,
    0.02,
    "ROR = Run-of-River",
    ha="center",
    fontsize=8
)

plt.tight_layout()

plot_file = "technology_effective_power_pie.png"
plt.savefig(plot_file, dpi=300, bbox_inches="tight")
plt.close(fig)

logging.info("Saved technology pie chart to: %s", plot_file)


INFO:Saved technology pie chart to: technology_effective_power_pie.png
